# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides an example for loading and analyzing the FAIR² dataset (Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
import pprint

# List all available record sets by their @id
print("Available record sets (use @id for access):")
recsets = list(dataset.record_sets)
for rs in recsets:
    print(f"  @id: {rs.id} | name: {rs.name if hasattr(rs, 'name') else ''}")

# For demonstration, list fields for the first record set
if recsets:
    example_record_set = recsets[0]
    print(f"\nFields in record set @id={example_record_set.id}:")
    for field in example_record_set.fields:
        print(f"  @id: {field.id} | name: {field.name}")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

**Note:** All notation (record set, field, column) must use their respective `@id` fields.

In [ ]:
# Step 1: Prepare a list of record set @id's
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record_set @id='{record_set_id}' with shape {df.shape}")

# List columns in the first available DataFrame for demonstration
if record_sets:
    example_record_set_id = record_sets[0]
    print(f"\nColumns in example record set (@id={example_record_set_id}):\n  {dataframes[example_record_set_id].columns.tolist()}")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, categorizing/grouping data, removing outliers, and transforming data distributions.

All field references must use their `@id`s. Variables are used to select fields dynamically.

In [ ]:
# For demonstration: identify numeric fields in the example DataFrame (by @id)
import numpy as np

df = dataframes[example_record_set_id]
numeric_field_ids = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric field @id's in example record set:", numeric_field_ids)

# Select a numeric field (by @id)
if numeric_field_ids:
    numeric_field_id = numeric_field_ids[0]  # e.g., '@id' of Age or similar
    print(f"\nUsing '{numeric_field_id}' for demonstration.")

    # Filter records with the numeric field above a threshold
    threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Add a normalized column
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, normalized_col]].head())

    # Try to pick a group (categorical) field
    cat_field_ids = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id]
    if cat_field_ids:
        group_field_id = cat_field_ids[0]
        print(f"\nGrouping by categorical field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(grouped_df.head())
else:
    print("No numeric fields detected in example DataFrame for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and the relationship with a categorical variable (all referenced via their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_ids:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if cat_field_ids:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[cat_field_ids[0]], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {cat_field_ids[0]}")
        plt.xlabel(cat_field_ids[0])
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to explore a FAIR² dataset defined by a Croissant schema. Using `@id` to reference all entities ensures unambiguous access to record sets and fields. This workflow supports robust, reproducible data science by strictly referencing semantic identifiers as per Croissant and FAIR best practices.

**Key findings and next steps:**
- The dataset consists of multiple record sets and rich metadata accessible via Croissant schema.
- Data was successfully loaded into pandas DataFrames using `mlcroissant`, with all entities referenced by `@id`.
- Numeric variables can be filtered, normalized, and visualized using familiar Python data analysis tools.
- Future work may include advanced analysis such as machine learning, survival analysis, or outcome modeling—all while maintaining strict semantic referencing via `@id`.